In [1]:
import requests

In [19]:
import requests
import json
from pathlib import Path
from datetime import datetime, timezone, timedelta


# ============================================================
# 1. 請填入你的 TDX Client ID / Client Secret
# ============================================================

CLIENT_ID = os.environ["TDX_CLIENT_ID"]
CLIENT_SECRET = os.environ["TDX_CLIENT_SECRET"]


# ============================================================
# 2. API 設定
# ============================================================

TOKEN_URL = (
    "https://tdx.transportdata.tw/"
    "auth/realms/TDXConnect/protocol/openid-connect/token"
)

API_URL = "https://tdx.transportdata.tw/api/basic/v2/Air/FIDS/Airport/TPE"

TAIWAN_TZ = timezone(timedelta(hours=8))


# ============================================================
# 3. 取得 Access Token
# ============================================================

def get_access_token():
    data = {
        "grant_type": "client_credentials",
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET
    }

    response = requests.post(
        TOKEN_URL,
        data=data,
        timeout=30
    )

    response.raise_for_status()

    token = response.json()["access_token"]

    return token


# ============================================================
# 4. 抓取桃園機場 FIDS 資料
# ============================================================

def get_tpe_flight_data(token):
    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/json"
    }

    params = {
        "$format": "JSON"
    }

    response = requests.get(
        API_URL,
        headers=headers,
        params=params,
        timeout=60
    )

    response.raise_for_status()

    return response.json()


# ============================================================
# 5. 儲存 JSON
# ============================================================

def save_json(data):
    now = datetime.now(TAIWAN_TZ)

    save_dir = Path("data/raw")
    save_dir.mkdir(parents=True, exist_ok=True)

    filename = save_dir / (
        f"tpe_fids_{now.strftime('%Y-%m-%d_%H%M%S')}.json"
    )

    with open(filename, "w", encoding="utf-8") as f:
        json.dump(
            data,
            f,
            ensure_ascii=False,
            indent=2
        )

    return filename


# ============================================================
# 6. 主程式
# ============================================================

def main():
    print("開始抓取桃園機場航班資料...")

    token = get_access_token()

    data = get_tpe_flight_data(token)

    filename = save_json(data)

    print("抓取完成")
    print(f"資料已儲存至：{filename}")


# ============================================================
# 7. 執行
# ============================================================

if __name__ == "__main__":
    main()

開始抓取桃園機場航班資料...
抓取完成
資料已儲存至：data\raw\tpe_fids_2026-09-28_230925.json


In [ ]:
https://tdx.transportdata.tw/api/basic/v2/Air/FIDS/Flight

In [18]:
import requests
from pathlib import Path
from datetime import datetime, timezone, timedelta


# ==========================================
# 1. 設定
# ==========================================

URL = "https://www.taoyuan-airport.com/uploads/flightx/a_flight_v4.txt"
# https://www.taoyuan-airport.com/uploads/flightx/af_flight_v4.txt


TAIWAN_TZ = timezone(timedelta(hours=8))


# ==========================================
# 2. 抓取資料
# ==========================================

def download_flight_data():
    headers = {
        "User-Agent": "Mozilla/5.0"
    }

    response = requests.get(
        URL,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()

    return response.content


# ==========================================
# 3. 儲存檔案
# ==========================================

def save_file(content):
    now = datetime.now(TAIWAN_TZ)

    save_dir = Path("data/raw")
    save_dir.mkdir(parents=True, exist_ok=True)

    filename = save_dir / (
        f"a_flight_v4_{now.strftime('%Y-%m-%d_%H%M%S')}.txt"
    )

    with open(filename, "wb") as f:
        f.write(content)

    return filename


# ==========================================
# 4. 主程式
# ==========================================

def main():
    print("開始抓取桃園機場航班資料...")

    content = download_flight_data()

    if len(content) < 1000:
        raise ValueError("下載到的資料量異常，可能不是正常航班資料")

    filename = save_file(content)

    print("抓取完成")
    print(f"檔案大小：{len(content)} bytes")
    print(f"儲存位置：{filename}")


# ==========================================
# 5. 執行
# ==========================================

if __name__ == "__main__":
    main()

開始抓取桃園機場航班資料...
抓取完成
檔案大小：33817 bytes
儲存位置：data\raw\af_flight_v4_2026-09-28_230909.txt
